# Does a ridge penalty help HAR?

HAR-ridge fits HAR's slopes by least squares plus a penalty
$\alpha\,\lVert b \rVert^2$, which pulls them towards zero and the forecast
towards the mean. Shrinkage often helps when estimates are noisy. This notebook
asks, for every asset, whether a penalty chosen without looking at the
evaluation period beats plain HAR.

Methods are explained in `02_har`. Every number that depends on the data is
printed by the code cell it comes from.

In [1]:
import _plots
import pandas as pd
from _common import CANDIDATE, LOSS, by_asset, pair, runs

from rvol import study

_plots.use_style()

## 1. Choosing the penalty

The penalty multiplies the squared slopes against the summed squared errors, so
the same $\alpha$ shrinks less as the training sample grows. It is chosen once
per asset, from a grid fixed in advance, on the calibration window: the
forecasts made before the evaluation starts. Each penalty is run walk-forward
over that window and scored with QLIKE exactly as the evaluation will score it;
the lowest score wins. The choice therefore sees no evaluation data, and the
grid is not widened after seeing the results, which would tune on the test
data.

In [2]:
RUNS = runs()
calibration = pd.DataFrame({pair(a): run.calibration for a, run in RUNS.items()})
for asset, run in RUNS.items():
    chosen = run.calibration.idxmin()
    edge = (
        " (the edge of the grid)"
        if chosen in (min(study.RIDGE_GRID), max(study.RIDGE_GRID))
        else ""
    )
    print(f"{pair(asset)}: chosen penalty {chosen:g}{edge}")
calibration.round(4)

EUR/USD: chosen penalty 100 (the edge of the grid)


,EUR/USD
ridge penalty,
0.1,0.3073
1.0,0.3073
10.0,0.3066
100.0,0.3031


## 2. Does it beat HAR?

The study tests every model against HAR, so here the DM p-value asks whether
HAR is better than HAR-ridge; a small value means the penalty hurt.

In [3]:
lower, wins = f"{CANDIDATE} lower by", f"{CANDIDATE} wins"
results = by_asset(
    {
        asset: study.results_table(run.comparison).loc[[CANDIDATE, "HAR-ridge"]]
        for asset, run in RUNS.items()
    }
)
for asset in RUNS:
    row = results.loc[(pair(asset), "HAR-ridge")]
    direction = "below" if row[lower] > 0 else "above"
    print(
        f"{pair(asset)}: {CANDIDATE}'s mean {LOSS} is {abs(row[lower]):.1%} {direction} "
        f"HAR-ridge's; one-sided p-value that {CANDIDATE} is better {row['DM p-value']:.1e}"
    )
results.style.format(
    {LOSS: "{:.4f}", lower: "{:.1%}", "DM p-value": "{:.1e}", wins: "{:.0%}"}, na_rep="—"
)

EUR/USD: HAR's mean QLIKE is 6.7% below HAR-ridge's; one-sided p-value that HAR is better 7.1e-04


## 3. With hindsight

Was the method wrong, or only the choice of penalty? Scoring every penalty in
the grid on the evaluation period answers that. This uses the evaluation data,
so it chooses nothing: it only shows what the calibration window could not.

In [4]:
def evaluation_loss(run, alpha):
    ridge = study.forecast(run.prepared, [study.ridge_har(alpha)], asset=run.asset)
    har = run.forecasts.query("model == @CANDIDATE")
    scored = study.compare(pd.concat([har, ridge]))
    return scored.summary.set_index("model").loc["HAR-ridge", "mean_loss"]


hindsight = pd.DataFrame(
    {
        pair(a): {alpha: evaluation_loss(run, alpha) for alpha in study.RIDGE_GRID}
        for a, run in RUNS.items()
    }
).rename_axis("ridge penalty")
for asset, run in RUNS.items():
    har = study.results_table(run.comparison).loc[CANDIDATE, LOSS]
    best = hindsight[pair(asset)].idxmin()
    print(
        f"{pair(asset)}: {CANDIDATE} without a penalty {har:.4f}; "
        f"best penalty with hindsight {best:g}, {hindsight.loc[best, pair(asset)]:.4f}"
    )
hindsight.round(4)

EUR/USD: HAR without a penalty 0.1525; best penalty with hindsight 0.1, 0.1525


,EUR/USD
ridge penalty,
0.1,0.1525
1.0,0.1525
10.0,0.1531
100.0,0.1635


## Answer

In [5]:
for asset, run in RUNS.items():
    chosen = run.calibration.idxmin()
    row = results.loc[(pair(asset), "HAR-ridge")]
    har = results.loc[(pair(asset), CANDIDATE), LOSS]
    best = hindsight[pair(asset)].min() / har - 1
    outcome = "loses to" if row[lower] > 0 else "beats"
    print(
        f"{pair(asset)}: with the chosen penalty {chosen:g}, HAR-ridge {outcome} {CANDIDATE} "
        f"(DM p-value {row['DM p-value']:.1e}); the best penalty in hindsight changes "
        f"{CANDIDATE}'s mean {LOSS} by {best:+.3%}"
    )

EUR/USD: with the chosen penalty 100, HAR-ridge loses to HAR (DM p-value 7.1e-04); the best penalty in hindsight changes HAR's mean QLIKE by +0.003%


If small penalties match HAR and the chosen one loses, the lesson is about
tuning, not about ridge: a few months of forecasts are too few to choose a
penalty, and a calibration score that keeps falling towards the edge of the
grid is a warning that the grid, or the window, cannot locate an optimum.

**Caveats.** The penalty is in units of summed squared regressors, so its
strength drifts as the training sample grows; a penalty per observation would
keep it constant and is a natural next variant.